# 01a · Zero-shot baseline: Chatterbox Multilingual

How close is Chatterbox Multilingual to real Derja **before** any fine-tuning?
We clone your voice from a 10-15 s clip and read real script lines twice: once raw, once through the derja-tts normalizer.

- Runtime: **GPU** (a free T4 is enough), about 10 minutes.
- Chatterbox is MIT-licensed, so it's the base we can actually release.
- Language token: `ar`. Chatterbox has no Tunisian token, so Arabic is the closest fit.
- Every output carries Resemble AI's PerTh watermark (inaudible).

In [ ]:
!pip install -q chatterbox-tts num2words
# If Colab asks to restart the session after this, do it, then continue from the next cell.

In [ ]:
REPO_URL = "https://github.com/<your-user>/derja-tts"   # change after you push
!git clone -q {REPO_URL} derja-tts || echo "Clone failed: upload the repo folder to /content/derja-tts instead"
import sys
sys.path.insert(0, "derja-tts/src")

In [ ]:
from google.colab import files
print("Upload 10-15 s of YOU speaking Derja (wav or mp3). Quiet room, no music, natural narration tone.")
uploaded = files.upload()
REF = next(iter(uploaded))
print("reference:", REF)

In [ ]:
import torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ChatterboxMultilingualTTS.from_pretrained(device=device)
print("device:", device, "| sample rate:", model.sr)

In [ ]:
from derja_tts.text import Normalizer
norm = Normalizer()

lines = [l.strip() for l in open("derja-tts/examples/sentences.txt", encoding="utf-8") if l.strip()]
# Add 2-3 lines copied from one of YOUR real video scripts, exactly as you wrote them:
lines += [
    # "...",
]

for i, line in enumerate(lines):
    r = norm.analyze(line)
    print(f"[{i:02d}] raw : {line}")
    print(f"     norm: {r.text}")
    if r.unknown:
        print(f"     unknown (kept Latin): {r.unknown}")

## Generate: raw vs normalized
`EXAGGERATION` controls expressiveness (0.25 calm, 1.0 dramatic). Lower `CFG_WEIGHT` to about 0.3 if your reference clip is fast.

In [ ]:
import os
from IPython.display import Audio, display

OUT = "out_chatterbox"
os.makedirs(OUT, exist_ok=True)
EXAGGERATION = 0.5
CFG_WEIGHT = 0.5

def say(text, path):
    wav = model.generate(text, language_id="ar", audio_prompt_path=REF,
                         exaggeration=EXAGGERATION, cfg_weight=CFG_WEIGHT)
    ta.save(path, wav, model.sr)
    return path

for i, line in enumerate(lines):
    normalized = norm(line)
    variants = [("norm", normalized)] if normalized == line else [("raw", line), ("norm", normalized)]
    for tag, text in variants:
        print(f"[{i:02d}] {tag}: {text}")
        display(Audio(say(text, f"{OUT}/{i:02d}_{tag}.wav")))

## Quick vocab check: ڨ vs ق
The normalizer writes Arabizi **g** as ڨ. If Chatterbox's tokenizer doesn't know ڨ, the word gets garbled. Listen to both. If only the second one works, use `NormalizerConfig(g_letter="ق")` (or `--g-letter ق` on the CLI).

In [ ]:
for tag, text in [("g_veh", "ڨال لي باهي، نبداو"), ("g_qaf", "قال لي باهي، نبداو")]:
    print(tag, text)
    display(Audio(say(text, f"{OUT}/{tag}.wav")))

## Score it
This sheet is the baseline. Every fine-tuned checkpoint later gets the same test.

In [ ]:
# Listen to every pair, then fill this in. 1 = bad, 5 = perfect.
#   tunisian   : does the Derja sound Tunisian (not MSA, not Levantine)?
#   tech       : are the English/French terms pronounced right?
#   voice      : does it sound like you?
#   better     : "raw", "norm" or "same"
scores = {
    # 0: dict(tunisian=3, tech=4, voice=4, better="norm", notes="overfitting OK, barcha too MSA"),
}

import csv
with open(f"{OUT}/scores.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["line", "raw", "norm", "tunisian", "tech", "voice", "better", "notes"])
    for i, s in sorted(scores.items()):
        w.writerow([i, lines[i], norm(lines[i]), s.get("tunisian"), s.get("tech"),
                    s.get("voice"), s.get("better"), s.get("notes", "")])
print("saved", f"{OUT}/scores.csv")

In [ ]:
!zip -qr {OUT}.zip {OUT}
files.download(f"{OUT}.zip")